# Week 2 Sensitivity Checks

This notebook supports two checks from Project Plan v9:

1. MAD-proxy vs textbook MAD Spearman rank correlation.
2. `min_liquidity` sweep over `{0.25, 0.5, 1.0, 2.0, 5.0}`.

In [3]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = next((candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / 'src').exists()), Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from market_anomaly_engine import MarketAnomalyEngine

WINDOW = 20
THRESHOLDS = (0.25, 0.5, 1.0, 2.0, 5.0)


def make_mock_price_fixture(
    n_tickers: int = 8,
    n_days: int = 200,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range('2024-01-02', periods=n_days)
    rows = []

    for ticker_idx in range(n_tickers):
        ticker = f'TKR{ticker_idx + 1:02d}'
        base_price = 20 + ticker_idx * 7
        base_volume = 150_000 + ticker_idx * 25_000
        returns = rng.normal(0.0005, 0.015, size=n_days)
        volume = rng.lognormal(mean=np.log(base_volume), sigma=0.25, size=n_days)

        spike_days = rng.choice(np.arange(20, n_days - 20), size=6, replace=False)
        for i, day in enumerate(spike_days):
            if i % 2 == 0:
                returns[day] += rng.choice([0.10, -0.09])
            else:
                returns[day] += rng.choice([0.06, -0.05])
            volume[day] *= rng.choice([0.15, 3.5, 5.0])

        prices = [base_price]
        for ret in returns[1:]:
            prices.append(prices[-1] * float(np.exp(ret)))

        frame = pd.DataFrame(
            {
                'Ticker': ticker,
                'Date': dates,
                'Adj_Close': prices,
                'Volume': volume,
            }
        )
        rows.append(frame)

    fixture = pd.concat(rows, ignore_index=True)
    fixture['Adj_Close'] = fixture['Adj_Close'].clip(lower=0.5)
    fixture['Volume'] = fixture['Volume'].clip(lower=1.0)
    return fixture.sort_values(['Ticker', 'Date']).reset_index(drop=True)


price_fixture = make_mock_price_fixture()
price_fixture.head()

,Ticker,Date,Adj_Close,Volume
0,TKR01,2024-01-02,20.000000,163208.566391
1,TKR01,2024-01-03,19.700273,213258.652379
2,TKR01,2024-01-04,19.933252,153435.689883
3,TKR01,2024-01-05,20.226583,176200.049145
4,TKR01,2024-01-08,19.653044,89845.566516


In [7]:
def textbook_mad_scores(frame: pd.DataFrame, window: int = WINDOW) -> pd.DataFrame:
    out = frame.copy()
    out = out.sort_values(["Ticker", "Date"]).reset_index(drop=True)

    def rolling_textbook_mad(series: pd.Series) -> pd.Series:
        return series.rolling(window).apply(
            lambda values: np.median(np.abs(values - np.median(values))),
            raw=True,
        )

    out["Log_Volume"] = np.log1p(out["Volume"])
    out["Volume_Textbook_MAD"] = out.groupby("Ticker", group_keys=False)["Log_Volume"].apply(rolling_textbook_mad)
    out["Volume_Textbook_Median"] = out.groupby("Ticker", group_keys=False)["Log_Volume"].transform(lambda series: series.rolling(window).median())
    out["Volume_Textbook_Z"] = (out["Log_Volume"] - out["Volume_Textbook_Median"]) / (1.4826 * out["Volume_Textbook_MAD"] + 1e-8)

    out["Returns"] = out.groupby("Ticker", group_keys=False)["Adj_Close"].transform(lambda series: np.log(series / series.shift(1)))
    out["Returns_Textbook_MAD"] = out.groupby("Ticker", group_keys=False)["Returns"].apply(rolling_textbook_mad)
    out["Returns_Textbook_Median"] = out.groupby("Ticker", group_keys=False)["Returns"].transform(lambda series: series.rolling(window).median())
    out["Returns_Textbook_Z"] = (out["Returns"] - out["Returns_Textbook_Median"]) / (1.4826 * out["Returns_Textbook_MAD"] + 1e-8)
    return out


def _spearman_rank_corr(left: pd.Series, right: pd.Series) -> float:
    return float(left.rank(method="average").corr(right.rank(method="average"), method="pearson"))


def spearman_proxy_vs_textbook(frame: pd.DataFrame, window: int = WINDOW) -> pd.DataFrame:
    engine = MarketAnomalyEngine(window=window)
    rows = []

    for ticker, ticker_frame in frame.groupby("Ticker", sort=True):
        proxy = engine.analyze_ticker_data(ticker_frame)
        textbook = textbook_mad_scores(ticker_frame, window=window)
        merged = proxy[["Ticker", "Date", "Vol_ZScore_Robust", "Volat_ZScore_Robust"]].merge(
            textbook[["Ticker", "Date", "Volume_Textbook_Z", "Returns_Textbook_Z"]],
            on=["Ticker", "Date"],
            how="inner",
        )

        volume_mask = merged["Vol_ZScore_Robust"].notna() & merged["Volume_Textbook_Z"].notna()
        returns_mask = merged["Volat_ZScore_Robust"].notna() & merged["Returns_Textbook_Z"].notna()

        rows.append(
            {
                "metric": "volume",
                "ticker": ticker,
                "spearman_r": _spearman_rank_corr(
                    merged.loc[volume_mask, "Vol_ZScore_Robust"].abs(),
                    merged.loc[volume_mask, "Volume_Textbook_Z"].abs(),
                ),
            }
        )
        rows.append(
            {
                "metric": "returns",
                "ticker": ticker,
                "spearman_r": _spearman_rank_corr(
                    merged.loc[returns_mask, "Volat_ZScore_Robust"].abs(),
                    merged.loc[returns_mask, "Returns_Textbook_Z"].abs(),
                ),
            }
        )

    return pd.DataFrame(rows)

In [8]:
def min_liquidity_sweep(frame: pd.DataFrame, thresholds=THRESHOLDS, window: int = WINDOW) -> pd.DataFrame:
    rows = []
    for threshold in thresholds:
        engine = MarketAnomalyEngine(window=window, min_liquidity=threshold)
        scored_frames = []
        for ticker, ticker_frame in frame.groupby("Ticker", sort=True):
            scored_frames.append(engine.analyze_ticker_data(ticker_frame))
        scored = pd.concat(scored_frames, ignore_index=True)

        volume_flagged_tickers = scored.loc[scored["Low_Liquidity_Volume_Flag"], "Ticker"].nunique()
        returns_flagged_tickers = scored.loc[scored["Low_Liquidity_Returns_Flag"], "Ticker"].nunique()

        rows.append(
            {
                "min_liquidity": threshold,
                "flagged_tickers_volume": int(volume_flagged_tickers),
                "flagged_tickers_returns": int(returns_flagged_tickers),
                "flagged_rows_volume": int(scored["Low_Liquidity_Volume_Flag"].sum()),
                "flagged_rows_returns": int(scored["Low_Liquidity_Returns_Flag"].sum()),
                "volume_anomalies": int(scored["Is_Volume_Anomaly"].sum()),
                "volatility_anomalies": int(scored["Is_Volatility_Anomaly"].sum()),
            }
        )

    return pd.DataFrame(rows)


spearman_summary = spearman_proxy_vs_textbook(price_fixture)
liquidity_summary = min_liquidity_sweep(price_fixture)

print("Synthetic fixture shape:", price_fixture.shape)
display(price_fixture.head(10))
display(spearman_summary)
display(liquidity_summary)

Synthetic fixture shape: (1600, 4)


,Ticker,Date,Adj_Close,Volume
0,TKR01,2024-01-02,20.000000,163208.566391
1,TKR01,2024-01-03,19.700273,213258.652379
2,TKR01,2024-01-04,19.933252,153435.689883
3,TKR01,2024-01-05,20.226583,176200.049145
4,TKR01,2024-01-08,19.653044,89845.566516
5,TKR01,2024-01-09,19.282530,148184.140596
6,TKR01,2024-01-10,19.329204,121489.486300
7,TKR01,2024-01-11,19.247352,110601.320561
8,TKR01,2024-01-12,19.252126,120433.436048
9,TKR01,2024-01-15,19.016857,137979.404866


,metric,ticker,spearman_r
0,volume,TKR01,0.987137
1,returns,TKR01,0.985691
2,volume,TKR02,0.979067
3,returns,TKR02,0.983264
4,volume,TKR03,0.977834
5,returns,TKR03,0.979523
6,volume,TKR04,0.994456
7,returns,TKR04,0.989731
8,volume,TKR05,0.985559
9,returns,TKR05,0.982068


,min_liquidity,flagged_tickers_volume,flagged_tickers_returns,flagged_rows_volume,flagged_rows_returns,volume_anomalies,volatility_anomalies
0,0.25,8,8,1236,1288,6,0
1,0.50,8,8,1296,1288,0,0
2,1.00,8,8,1296,1288,0,0
3,2.00,8,8,1296,1288,0,0
4,5.00,8,8,1296,1288,0,0


## Notes

- Use adjusted close, not raw close.
- Run this on 5-10 representative tickers for the Week 2 report.
- The synthetic fixture here uses 200 trading days so the two-pass MAD proxy can warm up cleanly.
- The percentile-rank fields are computed as expanding-window ranks, so they do not look ahead to future days.
- Record the Spearman correlation and threshold sweep table in the write-up.